In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

from pdm_learn.preprocessing import densitymap
from pdm_learn.simulation import build_heatmap_dataset, build_metric_dataset, collect_simulated_pairs, perturb_pair


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from bisect import bisect_left

In [ ]:
positive = pd.read_csv(DATA_DIR / 'simulated' / 'positive.csv').reset_index(drop=True)

In [ ]:
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression

SIM_CENTERS = np.linspace(-2, 2, 7)
EPSILON_STD = 3.0
NEGATIVE_REPEATS = 200
HEATMAP_SIGMA = 0.1
RANDOM_SEED = 42

def compute_mi(x, y):
    return mutual_info_regression(x.reshape(-1, 1), y, discrete_features=False)[0]

def bicor(x, y, c=9.0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = (~np.isnan(x)) & (~np.isnan(y))
    x = x[mask]
    y = y[mask]
    if len(x) < 3:
        return np.nan
    x_med = np.median(x)
    y_med = np.median(y)
    x_mad = np.median(np.abs(x - x_med))
    y_mad = np.median(np.abs(y - y_med))
    if x_mad == 0 or y_mad == 0:
        return np.nan
    ux = (x - x_med) / (c * x_mad)
    uy = (y - y_med) / (c * y_mad)
    wx = (1 - ux**2)**2
    wy = (1 - uy**2)**2
    wx[np.abs(ux) >= 1] = 0
    wy[np.abs(uy) >= 1] = 0
    xw = (x - x_med) * wx
    yw = (y - y_med) * wy
    numerator = np.sum(xw * yw)
    denominator = np.sqrt(np.sum(xw**2) * np.sum(yw**2))
    return numerator / denominator if denominator != 0 else np.nan

rng = np.random.default_rng(RANDOM_SEED)
preview_x = positive.iloc[60].to_numpy()
preview_y = positive.iloc[61].to_numpy()
preview_x, preview_y = perturb_pair(preview_x, preview_y, EPSILON_STD, centers=SIM_CENTERS, rng=rng)

plt.figure(figsize=(1, 1))
plt.scatter(preview_x, preview_y, s=2)
plt.xlim(min(SIM_CENTERS), max(SIM_CENTERS))
plt.ylim(min(SIM_CENTERS), max(SIM_CENTERS))
plt.xticks([])
plt.yticks([])
plt.show()

preview_map = densitymap(preview_x, preview_y, SIM_CENTERS, SIM_CENTERS, sigma=HEATMAP_SIGMA)[::-1]
plt.figure(figsize=(1, 1))
plt.imshow(preview_map, cmap='hot', interpolation='nearest')
plt.xticks([])
plt.yticks([])
plt.show()


In [ ]:
positive_pairs = collect_simulated_pairs(
    positive,
    repeats=1,
    epsilon_std=EPSILON_STD,
    centers=SIM_CENTERS,
    rng=np.random.default_rng(RANDOM_SEED),
)
negative_pairs = collect_simulated_pairs(
    positive,
    repeats=NEGATIVE_REPEATS,
    epsilon_std=EPSILON_STD,
    shuffle_y=True,
    centers=SIM_CENTERS,
    rng=np.random.default_rng(RANDOM_SEED + 1),
)

heatmap_positive = build_heatmap_dataset(
    positive,
    densitymap,
    centers=SIM_CENTERS,
    repeats=1,
    epsilon_std=EPSILON_STD,
    sigma=HEATMAP_SIGMA,
    simulated_pairs=positive_pairs,
)
heatmap_positive.to_csv(DATA_DIR / 'simulated' / 'positive_heatmap.csv', index=False)

heatmap_negative = build_heatmap_dataset(
    positive,
    densitymap,
    centers=SIM_CENTERS,
    repeats=NEGATIVE_REPEATS,
    epsilon_std=EPSILON_STD,
    sigma=HEATMAP_SIGMA,
    shuffle_y=True,
    simulated_pairs=negative_pairs,
)
heatmap_negative.to_csv(DATA_DIR / 'simulated' / 'negative_heatmap.csv', index=False)

heatmap_positive.head()


In [ ]:
metric_specs = {
    'pearson': lambda x, y: pearsonr(x, y)[0],
    'spearman': lambda x, y: spearmanr(x, y)[0],
    'mi': compute_mi,
    'bicor': bicor,
}

for offset, (name, metric_fn) in enumerate(metric_specs.items(), start=10):
    positive_df = build_metric_dataset(
        positive,
        metric_fn,
        repeats=1,
        epsilon_std=EPSILON_STD,
        centers=SIM_CENTERS,
        column_name=name.upper() if name != 'bicor' else 'BiCor',
        simulated_pairs=positive_pairs,
    )
    positive_df.to_csv(DATA_DIR / 'simulated' / f'positive_{name}.csv', index=False)

    negative_df = build_metric_dataset(
        positive,
        metric_fn,
        repeats=NEGATIVE_REPEATS,
        epsilon_std=EPSILON_STD,
        shuffle_y=True,
        centers=SIM_CENTERS,
        column_name=name.upper() if name != 'bicor' else 'BiCor',
        simulated_pairs=negative_pairs,
    )
    negative_df.to_csv(DATA_DIR / 'simulated' / f'negative_{name}.csv', index=False)

{name: pd.read_csv(DATA_DIR / 'simulated' / f'positive_{name}.csv').head() for name in metric_specs}


In [ ]:
pd.read_csv(DATA_DIR / 'simulated' / 'positive_pearson.csv').head()


In [ ]:
pd.read_csv(DATA_DIR / 'simulated' / 'positive_spearman.csv').head()


In [ ]:
pd.read_csv(DATA_DIR / 'simulated' / 'positive_mi.csv').head()


In [ ]:
pd.read_csv(DATA_DIR / 'simulated' / 'positive_bicor.csv').head()


In [ ]:
pd.read_csv(DATA_DIR / 'simulated' / 'negative_heatmap.csv').head()


In [ ]:
plt.imshow(positive_df.iloc[100, :].to_numpy().reshape(7,7), cmap='hot', interpolation='nearest')
plt.show()